In [99]:
import pandas as pd
import geopandas as gps

import geopandas as gpd
from shapely.geometry import box

## Construi datos dummies para gtfs
se poblan tablas necesarias en https://github.com/mrcagney/make_gtfs/tree/master?tab=readme-ov-file


- ``meta.csv`` (required). A CSV file containing network metadata.
  The CSV file contains the following columns.


- ``shapes.geojson`` (required). A GeoJSON file containing route shapes.
  The file comprises one feature collection of LineString features, where each feature's properties contains at least the attribute ``shape_id``.
  Each LineString should represent the run of one representive trip of a route.
  In particular, the LineString should not traverse the same section of road many times, unless you want a trip to actually do that.

- ``service_windows.csv`` (required). A CSV file containing service window
  information.


- ``frequencies.csv`` (required). A CSV file containing route frequency information.
  The CSV file contains the following columns.


  Missing speed values will be filled with values from the library's dictionary
  `SPEED_BY_RTYPE`.

- ``speed_zones.geojson`` (optional). A GeoJSON file of Polygons representing
  speed zones for routes.
  The file consists of one feature collection of Polygon features
  (in WGS84 coordinates), each with the properties

  - ``speed_zone_id`` (required): string; a unique identifier of the zone polygon; can
    be re-used if the polygon is re-used


- ``stops.csv`` (optional). A CSV file containing all the required
  and optional fields of ``stops.txt`` in
  `the GTFS <https://developers.google.com/transit/gtfs/reference/#stopstxt>`_.

## Parámetros

In [100]:
modo_prueba = False
fracc_size = 1

In [101]:
estado = "tampico"
ciudad = "tampico" # tampico
start_date = "20250101"
end_date = "20250102"

In [102]:
#path_base = "/Users/danielbustillos/Documents/ITDP/proyectos-ITDP/DatosGPS/Comparacion_tampico/make_gtfs/data/tampico/"
path_shapes = f"../1-scraping_ruta_directa/data/proc/rutas_procesadas_{ciudad}.geojson"
path_base_export = f"./data/proc/{ciudad}/"

In [103]:
# route types
# Crear columnas con valores repetidos
direction = 1           # 2 si usas ida y vuelta como placeholder personalizado
frequency_= 5            # 4 vehículos por hora
route_type = 2         # 2 = Rail, puedes cambiarlo a 3 para bus por ejemplo

In [104]:
# speed zones
speed = 15

In [105]:
# meta
speed_zone_type_3 = 30
speed_zone_type_2 = 30


## meta.csv

- ``meta.csv`` (required). A CSV file containing network metadata.
  The CSV file contains the following columns.

  - ``agency_name`` (required): string; the name of the transport
    agency
  - ``agency_url`` (required): string; a fully qualified URL for
    the transport agency
  - ``agency_timezone`` (required): string; timezone where the
    transit agency is located; timezone names never contain the
    space character but may contain an underscore; refer to
    `http://en.wikipedia.org/wiki/List_of_tz_zones <http://en.wikipedia.org/wiki/List_of_tz_zones>`_ for a list of valid values
  - ``start_date``, ``end_date`` (required): strings; the start
    and end dates for which all this network information is valid
    formated as YYYYMMDD strings


In [106]:
agency_name = ciudad + " agency"
agency_url = ciudad + "agency.com/" 
agency_timezone = "Pacific/Auckland"


# Construir los datos del agency.txt
data = {
    "agency_name": [f"{ciudad} agency"],
    "agency_url": [f"https://{ciudad.lower()}agency.com"],
    "agency_timezone": ["Pacific/Auckland"],
    "start_date": start_date, 
    "end_date": end_date, 
    "speed_route_type_3": speed_zone_type_3, 
    "speed_zone_type_2": speed_zone_type_2, 
}


In [107]:
# Crear el DataFrame
df_agency = pd.DataFrame(data)
df_agency.head()

,agency_name,agency_url,agency_timezone,start_date,end_date,speed_route_type_3,speed_zone_type_2
0,tampico agency,https://tampicoagency.com,Pacific/Auckland,20250101,20250102,30,30


In [108]:
# Guardar como archivo CSV (sin índice)
df_agency.to_csv(path_base_export + "meta.csv", index=False)

## service_windows
- ``service_windows.csv`` (required). A CSV file containing service window
  information.
  A *service window* is a time interval and a set of days of the
  week during which all routes have constant service frequency,
  e.g. Saturday and Sunday 07:00 to 09:00.
  The CSV file contains the following columns.

  - ``service_window_id`` (required): string; a unique identifier
    for a service window
  - ``start_time``, ``end_time`` (required): string; the start
    and end times of the service window in HH:MM:SS format where
    the hour is less than 24
  - ``monday``, ``tuesday``, ``wednesday``, ``thursday``,
    ``friday``, ``saturday``, ``sunday`` (required); 0
    or 1; indicates whether the service is active on the given day
    (1) or not (0)

In [109]:
# Parámetros
service_window_id = "weekday"
start_time = "06:00:00"
end_time = "11:00:00"
active_days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

In [110]:
# Construir el diccionario con los días activos
row = {
    "service_window_id": service_window_id,
    "start_time": start_time,
    "end_time": end_time,
    **{day: int(day in active_days) for day in ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]}
}

service_windows = pd.DataFrame([row])
service_windows.head()



,service_window_id,start_time,end_time,monday,tuesday,wednesday,thursday,friday,saturday,sunday
0,weekday,06:00:00,11:00:00,1,1,1,1,1,1,1


In [111]:
# Crear y guardar DataFrame
service_windows.to_csv(path_base_export + "service_windows.csv", index=False)

## Shapes.geojson 

``shapes.geojson`` (required). A GeoJSON file containing route shapes.
  The file comprises one feature collection of LineString features, where each feature's properties contains at least the attribute 

- shape_id (string): ID que será referenciado por los trips.txt.
- geometry: una LineString con coordenadas [lon, lat] que representan el trazo del viaje.

In [112]:

ruta_directa = gps.read_file(path_shapes)
ruta_directa = ruta_directa.drop_duplicates()

print(ruta_directa["data.route.shortName"].nunique())
ruta_directa.head()

121


,data.route.shortName,data.route.longName,shape_id,geometry
0,74,Isleta Pérez,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,Cascajal,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,Golfo,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,Altamira - Tampiquito por Soriana,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,Central Camionera - Madero,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


In [113]:
if modo_prueba:
    print("MODO PRUEBA ACTIVADO")
    size_rutas = ruta_directa.shape[0]
    head_size_sample = round(size_rutas * fracc_size)
    print(head_size_sample)
    ruta_directa = ruta_directa.head(head_size_sample)

In [114]:
shapes_file = ruta_directa.copy()
#shapes_file.drop(columns=["data.route.shortName", "data.route.longName"], inplace=True)
shapes_file.head()

,data.route.shortName,data.route.longName,shape_id,geometry
0,74,Isleta Pérez,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,Cascajal,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,Golfo,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,Altamira - Tampiquito por Soriana,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,Central Camionera - Madero,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


In [115]:
# Crear y guardar DataFrame
shapes_file.to_file(path_base_export + f"shapes.geojson", driver="GeoJSON")

In [128]:
shapes_file.head()

,data.route.shortName,data.route.longName,shape_id,geometry
0,74,Isleta Pérez,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,Cascajal,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,Golfo,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,Altamira - Tampiquito por Soriana,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,Central Camionera - Madero,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


In [129]:
print(shapes_file["data.route.shortName"].nunique())

121


## frequencies.csv
``frequencies.csv`` (required). A CSV file containing route frequency information.
  The CSV file contains the following columns.

  - ``route_short_name`` (required): string; a unique short name
    for the route, e.g. '51X'
  - ``route_long_name`` (required): string; full name of the route
    that is more descriptive than ``route_short_name``
  - ``route_type`` (required): integer; the
    `GTFS type of the route <https://developers.google.com/transit/gtfs/reference/extended-route-types>`_ 

| Código | Tipo de Ruta                          |
|--------|----------------------------------------|
| 0      | Tram, Streetcar, Light rail (Tranvía)  |
| 1      | Subway, Metro (Metro/Subterráneo)      |
| 2      | Rail (Tren convencional)               |
| 3      | Bus (Autobús)                          |
| 4      | Ferry (Transbordador/Barco)            |
| 5      | Cable car (Teleférico)                 |
| 6      | Gondola, Suspended cable car (Góndola) |
| 7      | Funicular                              |

  - ``service_window_id`` (required): string; a service window ID
    for the route taken from the file ``service_windows.csv``
  - ``direction`` (required): 0, 1, or 2; indicates
    whether the route travels in the direction of its shape (1), or in the reverse direction of its shape (0), or in both directions (2);
    in the latter case, trips will be created that travel in both
    directions along the route's shape, each direction operating at
    the given frequency;  otherwise, trips will be created that
    travel in only the given direction
  - ``frequency`` (required): integer; the frequency of the route
    during the service window in vehicles per hour.
  - ``shape_id`` (required): string; a shape ID that is listed in
    ``shapes.geojson`` and corresponds to the linestring of the
    (route, direction, service window) tuple
  - ``speed`` (optional): float; the average speed of the route in
    kilometers per hour

**Route types catalogo:**


In [130]:
# Obtener listas base
route_short_name = ruta_directa["data.route.shortName"]
route_long_name = ruta_directa["data.route.longName"]
shape_id = ruta_directa["shape_id"]

In [131]:
# Asegurar misma longitud
n = len(route_long_name)

# Crear columnas con valores repetidos
direction_list = [direction] * n            # 2 si usas ida y vuelta como placeholder personalizado
frequency_list = [frequency_] * n            # 4 vehículos por hora
route_type_list = [route_type] * n           # 3 = Rail, puedes cambiarlo a 3 para bus por ejemplo
service_window_list = [service_window_id] * n

# speed = 15 #km/h

In [132]:
# Construir DataFrame frequencies
freq_df = pd.DataFrame({
    "route_short_name": route_short_name,
    "route_long_name": route_long_name,
    "route_type": route_type_list,
    "service_window_id": service_window_id,
    "direction": direction_list,
    "frequency": frequency_list,
    "shape_id": ruta_directa["shape_id"],
})

# Agregar speed si se proporciona
if speed is not None:
    freq_df["speed"] = float(speed)

In [133]:
freq_df = freq_df[["route_short_name","route_long_name","route_type","shape_id","service_window_id","frequency","direction"]]

In [134]:
freq_df.head()

,route_short_name,route_long_name,route_type,shape_id,service_window_id,frequency,direction
0,74,Isleta Pérez,2,shape_74,weekday,5,1
1,76,Cascajal,2,shape_76,weekday,5,1
2,81,Golfo,2,shape_81,weekday,5,1
3,89A,Altamira - Tampiquito por Soriana,2,shape_89A,weekday,5,1
4,67,Central Camionera - Madero,2,shape_67,weekday,5,1


In [135]:
# Guardar CSV
freq_df.to_csv(path_base_export + "frequencies.csv", index=False)

# Ejemplo de uso
# generar_frequencies_csv(ruta_directa)

## Speed zones
``speed_zone_id`` (required): string; a unique identifier of the zone polygon; can
    be re-used if the polygon is re-used
  - ``route_type`` (required): integer; a GTFS route type to which the zone applies
  - ``speed`` (required): positive float; the average speed in kilometers per hour
    of routes of that route type that travel within the zone; overrides route
    speeds in ``frequencies.csv`` within the zone.

In [136]:
# Datos base
shape_id = "shape_basic_1"
speed_zone_id = "shape_basic_1"



# Construir DataFrame speed_zones
speed_zones = pd.DataFrame({
    "shape_id": [shape_id],
    "speed_zone_id": [speed_zone_id],
    "route_type": [route_type],
    "speed": [speed],
})

# Obtener bounds de geometrías en ruta_directa
bounds = ruta_directa.total_bounds  # [minx, miny, maxx, maxy]

# Crear el bounding box como Polygon
bounding_box = box(*bounds)

# Asignar la geometría al DataFrame
speed_zones_gdf = gpd.GeoDataFrame(speed_zones, geometry=[bounding_box], crs=ruta_directa.crs)

# Mostrar resultado
speed_zones_gdf

,shape_id,speed_zone_id,route_type,speed,geometry
0,shape_basic_1,shape_basic_1,2,15,"POLYGON ((-97.78684 22.20913, -97.78684 22.551..."


In [137]:
speed_zones_gdf.to_file(path_base_export + "speed_zones.geojson", driver="GeoJSON" )

In [138]:
path_base_export

'./data/proc/tampico/'